# Download dataset

In [ ]:
!kaggle datasets download -d shoumikgoswami/annotated-gmb-corpus -p .

In [ ]:
import zipfile
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from TorchCRF import CRF
import sys
from collections import defaultdict
from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_class_weight


import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cuda


In [ ]:
with zipfile.ZipFile("annotated-gmb-corpus.zip", "r") as zip_ref:
    zip_ref.extractall(".")  

## Load data

In [3]:
def load_data(filepath):
    df = pd.read_csv(filepath, delimiter="\t", encoding="utf-8")
    df = df.fillna(method="ffill")  
    return df

filepath = "GMB_dataset.txt"
df = load_data(filepath)
df.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_20308\4117693000.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill")


,Unnamed: 0,Sentence #,Word,POS,Tag
0,0,1.0,Thousands,NNS,O
1,1,1.0,of,IN,O
2,2,1.0,demonstrators,NNS,O
3,3,1.0,have,VBP,O
4,4,1.0,marched,VBN,O


## Preprocess data:
- Group the words and labels into sentences

In [ ]:
def preprocess_data(df):
    sentences = []
    tags = []
    sentence = []
    tag = []
    
    # iterate through rows and group words by sentence number
    for _, row in df.iterrows():
        sentence.append(row['Word'])
        tag.append(row['Tag'])
        
        if _ == len(df) - 1 or df.iloc[_ + 1]['Sentence #'] != row['Sentence #']:
            sentences.append(sentence)
            tags.append(tag)
            sentence = []
            tag = []
    
    return sentences, tags

sentences, labels = preprocess_data(df)
print(list(zip(sentences[0],labels[0])))


[('Thousands', 'O'), ('of', 'O'), ('demonstrators', 'O'), ('have', 'O'), ('marched', 'O'), ('through', 'O'), ('London', 'B-geo'), ('to', 'O'), ('protest', 'O'), ('the', 'O'), ('war', 'O'), ('in', 'O'), ('Iraq', 'B-geo'), ('and', 'O'), ('demand', 'O'), ('the', 'O'), ('withdrawal', 'O'), ('of', 'O'), ('British', 'B-gpe'), ('troops', 'O'), ('from', 'O'), ('that', 'O'), ('country', 'O'), ('.', 'O')]


- Create mapping

In [ ]:
word2idx = defaultdict(lambda: len(word2idx))
tag2idx = {'O': 0}  # start with 'O' tag

# create word2idx and tag2idx mappings
for sentence_tags in labels:
    for tag in sentence_tags:
        if tag not in tag2idx:
            tag2idx[tag] = len(tag2idx)

for sentence in sentences:
    for word in sentence:
        word2idx[word]  

word2idx = dict(word2idx)


- Prepare data with its correct word2idx and tag2idx mapping

In [16]:
def prepare_data(sentences, labels, word2idx, tag2idx):
    X = []
    y = []
    
    for sentence, sentence_tags in zip(sentences, labels):
        word_indices = [word2idx[word] for word in sentence]
        tag_indices = [tag2idx[tag] for tag in sentence_tags]
        X.append(word_indices)
        y.append(tag_indices)
    
    return X, y

X, y = prepare_data(sentences, labels, word2idx, tag2idx)
X_pad = pad_sequence([torch.tensor(x) for x in X], batch_first=True, padding_value=0)
y_pad = pad_sequence([torch.tensor(y) for y in y], batch_first=True, padding_value=tag2idx['O'])


### Prepare the dataloader

In [ ]:
class NERDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.long).to(device)
        y = torch.tensor(self.y[idx], dtype=torch.long).to(device)
        return x, y

# split data into training and testing (last 10% as test set)
train_size = int(0.9 * len(X))
train_dataset = NERDataset(X_pad[:train_size], y_pad[:train_size])
test_dataset = NERDataset(X_pad[train_size:], y_pad[train_size:])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


## Model definition:
- Use Bidirection LSTM.
- Add dropout before the fc layer to address overfitting
- Last layer is CRF
- Loss is computed with weighted class because this dataset has imbalanced class labels.

In [ ]:
class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim=128, hidden_dim=128,class_weights=None):
        self.class_weights=class_weights
        super(BiLSTM_CRF, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True, num_layers=1, dropout=0.3)
        self.dropout = nn.Dropout(0.3)  
        self.fc = nn.Linear(2 * hidden_dim, tagset_size)  # since LSTM is bidirectional
        self.crf = CRF(tagset_size)

    def forward(self, x, mask):
        embeddings = self.embedding(x)
        lstm_out, _ = self.lstm(embeddings)
        lstm_out = self.dropout(lstm_out)  
        emissions = self.fc(lstm_out) 
        return emissions

    def compute_loss(self, emissions, tags, mask):
        log_likelihood = self.crf(emissions, tags, mask=mask)  
        
        if self.class_weights is not None:
            weights = self.class_weights[tags]  
            
            seq_weights = (weights * mask.float()).sum(1) / mask.float().sum(1) 
            
            weighted_loss = -log_likelihood * seq_weights
            
            return weighted_loss.mean()
        else:
            return -log_likelihood.mean()


    def decode(self, emissions, mask):
        return self.crf.viterbi_decode(emissions, mask=mask)

- Compute weighted loss

In [52]:
all_tags = []
for _, y_batch in train_loader:
    all_tags.extend(y_batch.cpu().numpy().flatten())
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(all_tags),
    y=all_tags
)
class_weights = torch.FloatTensor(class_weights).to(device)  # Send to GPU if available


C:\Users\Admin\AppData\Local\Temp\ipykernel_20308\3672655296.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(self.X[idx], dtype=torch.long).to(device)
C:\Users\Admin\AppData\Local\Temp\ipykernel_20308\3672655296.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(self.y[idx], dtype=torch.long).to(device)


- Train model using adam optimizer.
- For each epoch: train and update loss, print the first sentence prediction and its true label as well as the test accuracy to keep track of convergence

In [56]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            mask = (X_batch != 0).bool()  
            
            emissions = model(X_batch, mask)
            preds = model.decode(emissions, mask)  
            
            for i in range(len(preds)):
                pred = preds[i]
                true_label = y_batch[i].tolist()
                m = mask[i].tolist()  
                
                pred_filtered = [p for p, m_ in zip(pred, m) if m_ != 0]  # Filtered predictions
                true_filtered = [t for t, m_ in zip(true_label, m) if m_ != 0]  # Filtered true labels
                
                all_preds.append(pred_filtered)
                all_labels.append(true_filtered)
    
    print("Predictions:", all_preds[:1])
    print("True Labels:", all_labels[:1])

    return all_preds, all_labels


def calculate_accuracy(predictions, true_labels):
    correct = 0
    total = 0
    for pred, true in zip(predictions, true_labels):
        correct += sum(p == t for p, t in zip(pred, true))
        total += len(true)
    return correct / total

def train_model(model, train_loader, optimizer, num_epochs=5):
    best_test_accuracy = 0
    epochs_without_improvement = 0
    patience = 5

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            mask = (X_batch != 0).bool()    # Masking padding tokens
            optimizer.zero_grad()
            
            emissions = model(X_batch, mask)
            loss = model.compute_loss(emissions, y_batch, mask)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        model.eval()
        all_preds, all_labels = evaluate_model(model, test_loader)
        
        # Calculate accuracy on the test set
        accuracy = calculate_accuracy(all_preds, all_labels)
        if accuracy > best_test_accuracy:
            best_test_accuracy = accuracy
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        # if epochs_without_improvement >= patience:
        #     print("Early stopping due to no improvement")
        #     break
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {total_loss / len(train_loader):.4f}, Test Accuracy: {accuracy:.4f}')
    print(f'Accuracy: {accuracy:.4f}  Best accuracy: {best_test_accuracy:.4f}')

        

model = BiLSTM_CRF(len(word2idx), len(tag2idx),class_weights=class_weights).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
train_model(model, train_loader, optimizer,50)


c:\Python312\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(
C:\Users\Admin\AppData\Local\Temp\ipykernel_20308\3672655296.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(self.X[idx], dtype=torch.long).to(device)
C:\Users\Admin\AppData\Local\Temp\ipykernel_20308\3672655296.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(self.y[idx], dtype=torch.long).to(device)


Predictions: [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
True Labels: [[2, 3, 10, 10, 0, 0, 0, 0, 3, 10, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
Epoch 1/50, Loss: 90.7437, Test Accuracy: 0.8438
Predictions: [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
True Labels: [[2, 3, 10, 10, 0, 0, 0, 0, 3, 10, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
Epoch 2/50, Loss: 61.5751, Test Accuracy: 0.8594
Predictions: [[0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
True Labels: [[2, 3, 10, 10, 0, 0, 0, 0, 3, 10, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
Epoch 3/50, Loss: 47.2916, Test Accuracy: 0.8814
Predictions: [[2, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 7, 0